# Aprendizado de Máquina — Lista prática 03

## Seleção de Modelos e Validação Cruzada

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

A Lista prática 01 terminou num impasse: escolher o grau pelo erro de um conjunto
de teste é uma decisão ruidosa, e o grau vencedor mudava de amostra para amostra.
Esta lista traz a ferramenta que resolve isso.

Você vai seguir o protocolo da aula, nos quatro passos do slide — **separa** o
teste, **escolhe** por validação cruzada dentro do treino, **reajusta** o vencedor
em todo o treino, **mede** no teste uma única vez. No caminho vai escrever as $k$
dobras à mão, conferir contra o `scikit-learn`, e cair na armadilha mais comum da
validação cruzada:

> **as dobras têm de ser aleatórias. Se os dados chegam ordenados e você não
> embaralha, a estimativa não fica um pouco pior — ela fica errada.**

Cada lacuna está marcada com `...`. Substitua **cada uma** pela sua resposta e
rode a célula.

---
## 1. Importando os pacotes

In [ ]:
import numpy as np
from matplotlib.pyplot import subplots

import sklearn.linear_model as skl
import sklearn.model_selection as skm
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

import warnings
warnings.filterwarnings("ignore")

---
## Exercício 1 — o teste guardado, e as dobras dentro do treino

Mesma população das listas anteriores, agora com $n=120$ e semente 2026.

O **primeiro** passo do protocolo é separar o conjunto de teste, e ele não volta a
aparecer até o Exercício 4. Tudo o que vier antes — as dobras, a escolha do
hiperparâmetro — acontece dentro do treino.

Repare na proporção: 25% de teste deixa 90 observações de treino, e 90 é múltiplo
de 5. Isso não é obsessão por número redondo — com dobras de tamanhos diferentes,
a fórmula do Exercício 2 deixa de coincidir com a média das dobras.

In [ ]:
rng = np.random.default_rng(2026)
n = 120

x = rng.uniform(-3, 3, size=n)
y = np.sin(1.5 * x) + 0.3 * x + rng.normal(0, 0.7, size=n)
X = x.reshape(-1, 1)

# (a) o teste sai primeiro, e fica guardado ate o Exercicio 4
X_tr, X_te, y_tr, y_te = skm.train_test_split(X, y, test_size=0.25, random_state=2026)

cv = skm.KFold(n_splits=5, shuffle=True, random_state=2026)   # (b) e (c)

tamanhos = [len(indices_te) for _, indices_te in cv.split(X_tr)]   # (d) dentro do TREINO
print("treino:", len(y_tr), " teste:", len(y_te))
print("tamanhos das dobras:", tamanhos)

Deve imprimir `treino: 90  teste: 30` e
`tamanhos das dobras: [18, 18, 18, 18, 18]`.

`cv.split(X_tr)` devolve, a cada iteração, o par (índices de treino, índices de
validação) — **dentro do conjunto de treinamento**. As 90 observações se repartem
em 5 lotes de 18, e cada uma aparece na validação **exatamente uma vez**. As 30 do
teste não entram em dobra nenhuma.

---
## Exercício 2 — as $k$ dobras à mão

Implemente a Equação da nota,

$$\widehat{R}_{k\text{-CV}}
  = \frac1n \sum_{j=1}^{k}\sum_{i \in L_j}\big(Y_i - g_{-j}(X_i)\big)^2,$$

usando um polinômio de grau 5. Para cada dobra: ajuste **sem** ela, preveja
**nela**, guarde o erro quadrático médio. No fim, tire a média das cinco.

Aqui $n$ é o tamanho do **treino**, 90 — o teste não participa.

In [ ]:
def tubo(grau):
    return Pipeline([
        ("poly", PolynomialFeatures(degree=grau, include_bias=False)),
        ("escala", StandardScaler()),
        ("mqo", skl.LinearRegression()),
    ])


erros = []
for indices_tr, indices_te in cv.split(X_tr):
    modelo = tubo(5).fit(X_tr[indices_tr], y_tr[indices_tr])    # (a) ajuste SEM a dobra
    pred = modelo.predict(X_tr[indices_te])                     # (b) preveja NA dobra
    erros.append(np.mean((y_tr[indices_te] - pred) ** 2))

a_mao = np.mean(erros)
print("erro por dobra:", [f"{e:.4f}" for e in erros])
print(f"CV a mao: {a_mao:.6f}")

Agora o mesmo pelo `scikit-learn`. Atenção à convenção: as funções de `scoring`
são sempre de **ganho** (quanto maior, melhor), então o erro quadrático aparece
negado, com o prefixo `neg_`.

In [ ]:
notas = skm.cross_val_score(tubo(5), X_tr, y_tr, cv=cv,
                            scoring="neg_mean_squared_error")  # (a)
do_sklearn = -notas.mean()                                     # (b) desfaça o sinal

print(f"cross_val_score: {do_sklearn:.6f}")
print(f"diferenca:       {abs(a_mao - do_sklearn):.2e}")

Deve imprimir `erro por dobra: ['0.8082', '0.7565', '0.3450', '0.9792',
'0.5479']`, `CV a mao: 0.687363`, `cross_val_score: 0.687363` e
`diferenca: 0.00e+00`.

Zero exato, não ``quase igual'': o `cross_val_score` faz literalmente o laço que
você escreveu. E é zero exato **porque as cinco dobras têm o mesmo tamanho** — se
tivessem tamanhos diferentes, somar os $n$ erros e dividir por $n$ não daria o
mesmo que a média das cinco médias.

Vale guardar a dispersão entre as dobras — de $0{,}35$ a $0{,}98$, quase o triplo.
A média das cinco é bem mais estável que qualquer uma delas, e é por isso que a
validação cruzada bate o *data splitting* simples **dentro do treino**.

---
## Exercício 3 — a armadilha do `shuffle`

Bancos reais quase nunca chegam em ordem aleatória: vêm ordenados por data, por
região, por identificador do cliente. Vamos simular isso ordenando o **treino** por
$x$ e rodando a CV **sem** embaralhar.

In [ ]:
ordem = np.argsort(X_tr.ravel())                               # (a) ordena por x
X_ord, y_ord = X_tr[ordem], y_tr[ordem]

sem_shuffle = -skm.cross_val_score(
    tubo(5), X_ord, y_ord,
    cv=skm.KFold(5, shuffle=False),                            # (b)
    scoring="neg_mean_squared_error").mean()

com_shuffle = -skm.cross_val_score(
    tubo(5), X_ord, y_ord,
    cv=skm.KFold(5, shuffle=True, random_state=2026),
    scoring="neg_mean_squared_error").mean()

print(f"sem shuffle: {sem_shuffle:.4f}")
print(f"com shuffle: {com_shuffle:.4f}")
print(f"razao:       {sem_shuffle / com_shuffle:.1f}x")         # (c)

Deve imprimir `sem shuffle: 2.2049`, `com shuffle: 0.6951` e `razao: 3.2x`.

Os **mesmos dados** e o **mesmo modelo** produzem estimativas de risco que
diferem por um fator de mais de 3, só por causa de um argumento booleano.

Para ver de onde vem o estrago, imprima o erro de **cada dobra** no caso sem
embaralhar.

In [ ]:
por_dobra = []
for indices_tr, indices_te in skm.KFold(5, shuffle=False).split(X_ord):
    modelo = tubo(5).fit(X_ord[indices_tr], y_ord[indices_tr])
    por_dobra.append(np.mean((y_ord[indices_te] - modelo.predict(X_ord[indices_te])) ** 2))
    print(f"dobra: x de {X_ord[indices_te].min():6.2f} a {X_ord[indices_te].max():6.2f}"
          f"   EQM {por_dobra[-1]:8.3f}")

Deve imprimir:

```
dobra: x de  -2.99 a  -1.32   EQM    1.017
dobra: x de  -1.31 a  -0.20   EQM    0.726
dobra: x de  -0.18 a   0.63   EQM    0.492
dobra: x de   0.64 a   1.65   EQM    1.028
dobra: x de   1.73 a   2.75   EQM    7.762
```

O diagnóstico salta aos olhos: **a última dobra sozinha responde por quase todo o
erro** — 7,76 contra 0,49 a 1,03 das outras. Com os dados ordenados, cada dobra é
um intervalo contíguo de $x$, e o modelo treinado sem ela nunca viu aquela região.
Nas dobras do meio ele interpola — e vai bem. Nas das pontas ele precisa
**extrapolar** um polinômio de grau 5 para fora do intervalo observado, que é a
coisa que polinômio faz de pior. Que a ponta direita doa mais que a esquerda é
acaso desta amostra; o que não é acaso é serem as pontas.

Com `shuffle=True` cada dobra é uma amostra aleatória de todo o intervalo, e
nenhum ajuste precisa extrapolar.

**A regra prática:** use `shuffle=True` a menos que a ordem signifique alguma
coisa. Quando ela significa — séries temporais, ou várias medições do mesmo
paciente — embaralhar é pior ainda, porque vaza informação entre treino e
validação. Aí o certo é `TimeSeriesSplit` ou `GroupKFold`, que é assunto da Aula 06.

> **Sua vez.** Ordene o treino por $y$ em vez de por $x$ e repita. O estrago é
> maior ou menor? Por quê?

---
## Exercício 4 — escolhendo o $k$ do KNN, e depois medindo

Agora o uso para o qual a validação cruzada existe: escolher um hiperparâmetro.
O `GridSearchCV` percorre a grade, roda a CV em cada ponto e guarda o vencedor —
tudo **dentro do treino**, que é onde o `.fit()` recebe `X_tr, y_tr`.

Note que o `Pipeline` inteiro vai para dentro da busca — a padronização é
reajustada em cada dobra, e não uma vez só no começo. É a disciplina da Aula 06,
adiantada aqui porque custa uma linha.

In [ ]:
tubo_knn = Pipeline([("escala", StandardScaler()),
                     ("knn", KNeighborsRegressor())])

grade = {"knn__n_neighbors": np.arange(1, 41)}                 # (a) o nome do passo, dois underscores

busca = skm.GridSearchCV(tubo_knn, grade, cv=cv,
                         scoring="neg_mean_squared_error").fit(X_tr, y_tr)   # (b) so o treino

print("melhor k:", busca.best_params_["knn__n_neighbors"])
print(f"minimo da CV no treino: {-busca.best_score_:.4f}")     # (c)

Deve imprimir `melhor k: 14` e `minimo da CV no treino: 0.7209`.

Guarde esse $0{,}7209$ com desconfiança: ele é o **menor** de quarenta estimativas
ruidosas, e um mínimo de várias variáveis aleatórias é enviesado para baixo. Serviu
para escolher; não serve para reportar. O número que se reporta vem no fim deste
exercício.

O `GridSearchCV` guarda a média e o desvio-padrão entre dobras de **todos** os
pontos da grade, em `cv_results_`. Use isso para aplicar a regra de um
erro-padrão.

Lembre que, no KNN, **$k$ maior é modelo mais simples**: a média é tirada sobre
mais vizinhos, e a curva fica mais lisa.

In [ ]:
ks = np.arange(1, 41)
media = -busca.cv_results_["mean_test_score"]
ep = busca.cv_results_["std_test_score"] / np.sqrt(5)          # (a) erro-padrao da media de 5 dobras

j = int(np.argmin(media))                                      # (b) posicao do minimo
limite = media[j] + ep[j]
dentro = ks[media <= limite]

print(f"minimo em k = {ks[j]} ({media[j]:.4f}), EP = {ep[j]:.4f}")
print(f"limite de 1 EP = {limite:.4f}")
print(f"k dentro da faixa: de {dentro.min()} a {dentro.max()}")
print(f"regra de 1 EP escolhe k = {int(dentro.max())}")        # (c) o mais SIMPLES da faixa

In [ ]:
fig, ax = subplots(figsize=(5.5, 3.2))
ax.plot(ks, media, lw=1.5)
ax.fill_between(ks, media - ep, media + ep, alpha=0.2)
ax.axhline(limite, ls="--", lw=1, color="gray")
ax.axvline(ks[j], ls=":", lw=1, color="gray")
ax.set_xlabel("$k$ (vizinhos)")
ax.set_ylabel("EQM estimado por CV")
fig.tight_layout()

Deve imprimir:

```
minimo em k = 14 (0.7209), EP = 0.1076
limite de 1 EP = 0.8285
k dentro da faixa: de 4 a 23
regra de 1 EP escolhe k = 23
```

Vinte dos quarenta valores de $k$ estão dentro de um erro-padrão do mínimo. A
curva é **plana** de $k=4$ a $k=23$, e a diferença entre escolher 14 e escolher 23
é menor que o ruído da própria estimativa.

Isso não é uma falha do método — é a informação mais útil que ele podia dar. A
leitura correta não é ``o $k$ ótimo é 14'', e sim ``qualquer $k$ entre 4 e 23
serve, e $k=1$ é claramente ruim ($1{,}3044$, quase o dobro do mínimo)''. A regra
de um erro-padrão apenas transforma essa leitura numa escolha automática,
preferindo o extremo mais simples da faixa.

### Os dois últimos passos

A CV terminou o serviço dela: entregou dois candidatos, $k=14$ pelo mínimo e
$k=23$ pela regra de 1-EP. Faltam os passos 3 e 4 do protocolo — **reajustar cada
um em todo o conjunto de treinamento** e só então **medir no teste**, que não foi
tocado desde o Exercício 1.

In [ ]:
for nome, k_esc in [("minimo da CV", int(ks[j])), ("regra de 1-EP", int(dentro.max()))]:
    final = Pipeline([("escala", StandardScaler()),
                      ("knn", KNeighborsRegressor(n_neighbors=k_esc))])
    final.fit(X_tr, y_tr)                                      # (a) passo 3: TODO o treino
    erro2 = (y_te - final.predict(X_te)) ** 2                  # (b) passo 4: o teste, enfim
    print(f"k = {k_esc:2d} ({nome:13s}): EQM no teste {erro2.mean():.4f}"
          f"   +/- {erro2.std(ddof=1) / np.sqrt(len(erro2)):.4f}")

Deve imprimir:

```
k = 14 (minimo da CV ): EQM no teste 0.4623   +/- 0.1019
k = 23 (regra de 1-EP): EQM no teste 0.4708   +/- 0.1018
```

Três leituras, e a terceira é a que mais ensina.

**1.** Os dois candidatos são indistinguíveis no teste: $0{,}0085$ de diferença,
contra um erro-padrão de $0{,}10$ — doze vezes maior. O teste confirma o que a CV
já dizia, e não desempata melhor que ela.

**2.** O EQM no teste ($0{,}46$) ficou bem abaixo do mínimo da CV ($0{,}72$). Não é
que o modelo tenha melhorado ao ser reajustado — é que $0{,}72$ era o mínimo de
quarenta estimativas ruidosas, e $0{,}46$ é uma medição só, com 30 pontos.

**3.** E $0{,}46$ está **abaixo de $\sigma^2 = 0{,}49$**, que é o piso teórico: com
30 observações de teste nenhum modelo pode ter risco menor que o erro irredutível.
O que aconteceu foi sorte amostral — e o $\pm 0{,}10$ ao lado do número diz
exatamente isso. É a diferença entre o erro de treino, que fica abaixo de
$\sigma^2$ por **construção**, e uma medição honesta, que fica por **acaso** e
avisa o tamanho do acaso.

> **Sua vez.** Rode o `GridSearchCV` de novo trocando o `random_state` do `KFold`
> por 0, 1 e 2. Quanto o $k$ escolhido pelo mínimo varia entre as três rodadas? E
> o $k$ escolhido pela regra de 1 EP?

---
## O que ficou

| Exercício | O que você mediu |
|---|---|
| 1 | o teste sai primeiro e fica guardado; as dobras se cortam **dentro** do treino |
| 2 | as $k$ dobras à mão e o `cross_val_score` dão o mesmo número, com diferença 0 |
| 2 | as 5 dobras individuais variam de 0,35 a 0,98 — a média é bem mais estável |
| 3 | com os dados ordenados e `shuffle=False`, a estimativa fica **3,2×** maior |
| 3 | o estrago está nas dobras das pontas, que forçam extrapolação |
| 4 | o mínimo cai em $k=14$, e 20 dos 40 valores estão dentro de um erro-padrão |
| 4 | o mínimo da CV ($0{,}72$) não é o desempenho: o teste diz $0{,}46 \pm 0{,}10$ |

**A seguir.** A Aula 04 troca o polinômio global por métodos que olham só a
vizinhança do ponto — e o $k$ que você acabou de escolher passa a ser o
hiperparâmetro principal.